[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amaciasagro/GIT-RemoteSensing/blob/master/Climate/Climate_td_v2.ipynb)

# 🌦️ Climate Analyzer — v2
**Autor:** Ariel Macías | Agrónomo · GIS & Remote Sensing

Análisis climático sobre un lote agrícola usando **ERA5-Land** (ECMWF) via Google Earth Engine.

| Celda | Descripción |
|-------|-------------|
| 0 | Configuración (editar `config.py`) |
| 1 | Inicializar GEE y dibujar lote |
| 2 | Descarga ERA5 + métricas agronómicas + gráficos |

### Variables extraídas de ERA5-Land
| Variable | Descripción |
|----------|-------------|
| Precipitación | Suma diaria (mm) |
| T. máx / mín / media | Temperatura a 2 m (°C) |
| Punto de rocío | → Humedad Relativa estimada (%) |
| Radiación solar | Descendente superficial (MJ/m²/día) |
| Viento (u, v) | Componentes a 10 m → velocidad escalar (m/s) |
| ET vegetación | Evapotranspiración real ERA5 (mm) |
| **ETo Penman-Monteith** | Calculada con FAO-56 a partir de las anteriores |
| **Balance hídrico** | Lluvia acum. − ETo acum. (mm) |
| **GDA** | Grados día acumulados sobre T_BASE (config.py) |

In [ ]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# Editá estas variables antes de correr el notebook.
# Para mayor seguridad podés cargar el proyecto desde entorno:
#   import os; GEE_PROJECT = os.environ.get('GEE_PROJECT', 'tu-proyecto')
# ============================================================

GEE_PROJECT  = 'my-project'   # <-- Reemplazá con tu project ID de GEE
CENTRO_LAT   = 33.584              # Latitud del centro del mapa inicial
CENTRO_LON   = -101.845            # Longitud del centro del mapa inicial
ZOOM_INICIAL = 14

print("✅ Configuración cargada.")
print(f"   Proyecto GEE : {GEE_PROJECT}")
print(f"   Centro mapa  : ({CENTRO_LAT}, {CENTRO_LON})")

---
## Paso 1 · Inicializar GEE y dibujar el lote
Usá la herramienta de polígono (panel izquierdo) para delimitar el lote. Luego ejecutá la Celda 2.

In [ ]:
# ============================================================
# CELDA 1 — INICIALIZACIÓN GEE Y DIBUJO / CARGA DEL LOTE
# ============================================================
import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from IPython.display import display, HTML, FileLink
from contextlib import redirect_stdout
import json
import tempfile
import os

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# --- Autenticación GEE ---
try:
    ee.Initialize(project='my-project')
    print("✅ Earth Engine inicializado.")
except Exception:
    print("🔑 Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project='my-project')
    print("✅ Earth Engine inicializado.")

# ── Variable compartida entre ambos flujos ──────────────────
lote_geom      = None   # ee.Geometry — usada por celdas siguientes
lote_geom_shp  = None   # shapely geometry

# ============================================================
# MAPA INTERACTIVO
# ============================================================
Draw_Map = geemap.Map(center=[CENTRO_LAT, CENTRO_LON], zoom=ZOOM_INICIAL)
Draw_Map.add_basemap('SATELLITE')

# ── Botón: Confirmar polígono dibujado ──────────────────────
btn_confirmar = widgets.Button(
    description='✅ Confirmar polígono dibujado',
    button_style='success',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_dibujo = widgets.Output()

def confirmar_poligono(b):
    global lote_geom, lote_geom_shp
    with out_dibujo:
        out_dibujo.clear_output()
        roi = Draw_Map.user_roi
        if roi is None:
            print("⚠️ No se detectó ningún polígono. Dibujá uno primero.")
            return

        roi_info  = roi.getInfo()
        lote_geom = ee.Geometry(roi_info.get('geometry', roi_info))
        coords    = lote_geom.coordinates().getInfo()[0]
        lote_geom_shp = Polygon(coords)

        print(f"✅ Polígono confirmado — {len(coords)-1} vértices")
        print(f"   Área aprox.: {lote_geom_shp.area * 111320**2 / 10000:.2f} ha")

        # ── Exportar como GeoJSON descargable ──
        geojson_str = json.dumps({
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "geometry": mapping(lote_geom_shp),
                "properties": {"nombre": "lote"}
            }]
        }, indent=2)

        geojson_path = os.path.join(tempfile.gettempdir(), "lote.geojson")
        with open(geojson_path, "w") as f:
            f.write(geojson_str)

        # Copiar a directorio de trabajo para que el link funcione
        local_path = "lote_exportado.geojson"
        with open(local_path, "w") as f:
            f.write(geojson_str)

        print(f"\n📥 Descargá el polígono:")
        display(FileLink(local_path, result_html_prefix="👉 "))

btn_confirmar.on_click(confirmar_poligono)

# ============================================================
# CARGA DE SHAPEFILE LOCAL
# ============================================================
uploader = widgets.FileUpload(
    description='📂 Subir .shp',
    accept='.shp,.zip',   # .zip con los componentes del SHP
    multiple=True,        # necesario para .shp + .dbf + .shx + .prj
    layout=widgets.Layout(width='260px', margin='4px')
)
btn_cargar = widgets.Button(
    description='📌 Cargar SHP al mapa',
    button_style='info',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_shp = widgets.Output()

def cargar_shp(b):
    global lote_geom, lote_geom_shp
    with out_shp:
        out_shp.clear_output()
        archivos = uploader.value

        if not archivos:
            print("⚠️ No se subió ningún archivo.")
            return

        tmp_dir = tempfile.mkdtemp()

        # ── Compatible con ipywidgets >= 8 (value es tupla de dicts) ──
        for archivo in archivos:
            nombre   = archivo['name']
            contenido = archivo['content']
            ruta = os.path.join(tmp_dir, nombre)
            with open(ruta, 'wb') as f:
                f.write(contenido)

        # Buscar .zip o .shp
        zip_files = [f for f in os.listdir(tmp_dir) if f.endswith('.zip')]
        shp_files = [f for f in os.listdir(tmp_dir) if f.endswith('.shp')]

        try:
            if zip_files:
                import zipfile
                zip_path = os.path.join(tmp_dir, zip_files[0])
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(tmp_dir)
                shp_files = [f for f in os.listdir(tmp_dir) if f.endswith('.shp')]

            if not shp_files:
                print("❌ No se encontró .shp. Subí el .zip con todos los componentes.")
                return

            shp_path = os.path.join(tmp_dir, shp_files[0])
            gdf = gpd.read_file(shp_path).to_crs(epsg=4326)

            lote_geom_shp = unary_union(gdf.geometry)
            geojson_geom  = mapping(lote_geom_shp)
            lote_geom     = ee.Geometry(geojson_geom)

            print(f"✅ SHP cargado: {len(gdf)} feature(s) — reproyectado a EPSG:4326")
            print(f"   Área aprox.: {lote_geom_shp.area * 111320**2 / 10000:.2f} ha")

            with redirect_stdout(io.StringIO()):
                Draw_Map.add_gdf(
                    gdf,
                    layer_name="Lote (SHP)",
                    style={'color': '#f1c40f', 'weight': 2, 'fillOpacity': 0.2}
                )
            print("🗺️  Lote agregado al mapa.")

        except Exception as e:
            print(f"❌ Error al leer el archivo: {e}")

btn_cargar.on_click(cargar_shp)

# ============================================================
# LAYOUT FINAL
# ============================================================
titulo = widgets.HTML("<h3 style='margin:8px 0'>📍 Definición del lote</h3>")

panel_dibujo = widgets.VBox([
    widgets.HTML("<b>Opción A — Dibujar en el mapa</b><br><small>Usá el polígono del panel izquierdo</small>"),
    btn_confirmar,
    out_dibujo
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='4px', width='300px'))

panel_shp = widgets.VBox([
    widgets.HTML("<b>Opción B — Subir Shapefile</b><br><small>Subí el .zip (shp+dbf+shx+prj) o los archivos sueltos</small>"),
    uploader,
    btn_cargar,
    out_shp
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='4px', width='600px'))

display(titulo)
display(Draw_Map)
display(widgets.HBox([panel_dibujo, panel_shp]))

print("\n💡 Tip: podés dibujar Y también cargar un SHP — el último confirmado queda activo como 'lote_geom'.")

---
## Paso 2 · Descarga ERA5 · Métricas agronómicas · Gráficos
Descarga datos ERA5-Land para el lote dibujado, calcula ETo (Penman-Monteith FAO-56),
balance hídrico y grados día acumulados, y genera los paneles interactivos.

In [ ]:
# ============================================================
# CELDA 2 — ANÁLISIS COMPLETO
# ============================================================
from shapely.geometry import shape
import pandas as pd

from gee_utils      import obtener_fecha_maxima, calcular_periodos, descargar_serie
from agro_metrics   import calcular_eto_pm, calcular_hr, agregar_mensual, procesar_diario
from plots          import mostrar_graficos
from config         import ERA5_COLLECTION, T_BASE

# 1. Verificar dibujo
if lote_geom is None:
    raise ValueError("⚠️ No se definió ningún lote. Volvé a la Celda 1, dibujá o subí un archivo y confirmá.")

print("📐 Configurando geometría del lote...")
punto_clima = lote_geom  # área completa del lote, no centroide

# 2. Geometría del lote (área completa, no centroide)
# Usar la geometría ya confirmada (dibujada o cargada)
centroid = lote_geom.centroid(maxError=1).coordinates().getInfo()
latitud   = centroid[1]   # necesaria para ETo PM

# 3. Fechas automáticas
era5_base = ee.ImageCollection(ERA5_COLLECTION).filterBounds(lote_geom)
fecha_max = obtener_fecha_maxima(era5_base)
hist_start, hist_end, daily_start, daily_end = calcular_periodos(fecha_max)

print(f'📅 Histórico  : {hist_start} → {hist_end}')
print(f'📅 Mes actual : {daily_start} → {daily_end}')

# 4. Descarga ERA5 (area mean sobre el lote, no centroide)
df = descargar_serie(lote_geom, hist_start, hist_end)

# 5. Métricas agronómicas
df['eto'] = calcular_eto_pm(df, latitud)
df['hr']  = calcular_hr(df)

# 6. Agregación mensual
df_mensual = agregar_mensual(df)

# 7. Detalle diario del mes actual
df_diario = procesar_diario(df, daily_start, daily_end)
df_diario.attrs['t_base'] = T_BASE   # para label del gráfico

# 8. Métricas resumen (imprimir antes de los gráficos)
mes_actual = df_mensual.iloc[-1]
print(f'\n📊 Resumen del último mes disponible ({mes_actual["mes_año"].strftime("%b %Y")})')
print(f'   Lluvia      : {mes_actual["precip"]:.1f} mm')
print(f'   ETo PM      : {mes_actual["eto"]:.1f} mm')
print(f'   Balance     : {mes_actual["balance_hidro"]:+.1f} mm')
print(f'   T. media    : {mes_actual["t_med"]:.1f} °C')
print(f'   HR media    : {mes_actual["hr"]:.0f} %')

# 9. Gráficos
print('\n📈 Generando gráficos interactivos...')
mostrar_graficos(df_mensual, df_diario, daily_start, daily_end)